In [15]:
# notebooks/test_data.ipynb
# 1. Autoreload 설정
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parents[0]))

from data import fmri
from data.fmri import FmriVoxelData 

import numpy as np

# -----------------------------
# 1) 가짜(faked) fMRI 데이터 만들기
# -----------------------------
S, V, T = 3, 10000, 500          # 사람 2명, 복셀 5개, 시간 10개
C = 3                       # 좌표는 (x,y,z) 3차원이라고 가정

data = np.random.randn(S, V, T)      # (2, 5, 10)
voxel_coord = np.random.randint(0, 100, size=(V, C))  # (5, 3)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# 1. Validation 만 수행 (아직 data 연산 안함)
raw = FmriVoxelData(data=data, voxel_coord=voxel_coord).validate()

In [17]:
# 2. 객체 생성과 동시에 내부에서 gaussian_smoothing_1d 실행
prep = raw.preprocess("gaussian_smoothing", {"fwhm_samples": 3.0, "n_jobs": -1})

In [18]:
# 3. 그 다음 단계로 전이
sim = prep.compute_similarity(
    "correlation",
    {
        "chunk_size": 50000,
        "n_jobs": 8,
        "r_min": 0.1,
        "clamp_negative": True,
    },
)

In [19]:
# 4. 클러스터링 객체 생성
clu = sim.cluster(
    cluster_method="spectral",
    cluster_hyper={
        "n_clusters": 20,         # 필수
        "remove_isolated": True,
        "kmeans_nstart": 10,
        "kmeans_itermax": 200,
        "random_state": 2026,
        "eig_solver": "auto",         # 대규모면 randomized 자동 전환
        "randomized_switch_n_nodes": 15000,
        "randomized_oversample": 20,
        "randomized_n_iter": 2,
    },
)

In [20]:
from viz import plot_all_subject_clusters_3d

figs = plot_all_subject_clusters_3d(
    clu,
    include_unassigned=False,  # -1/0 라벨 제외 시각화
    marker_size=2.0,
    marker_opacity=0.85,
)

for subject_index, fig in enumerate(figs):
    print(f"subject {subject_index}: rendering cluster figure")
    fig.show()

subject 0: rendering cluster figure


subject 1: rendering cluster figure


subject 2: rendering cluster figure


In [21]:
import numpy as np
from viz import build_silhouette_report

sil_report = build_silhouette_report(clu)

print("per_subject silhouette:", sil_report["per_subject"])
print("mean silhouette:", sil_report["mean"])
print("sd silhouette:", sil_report["sd"])

# subject별 cluster 상세 지표(a_k, b_k, s_k, n_k)
for s, detail in enumerate(sil_report["per_subject_details"]):
    print(f"\n[subject {s}] overall={detail['overall']}")
    print("clusters:", sorted(detail["per_cluster"].keys()))

per_subject silhouette: [0.5456866208876072, 0.5459056584012325, 0.5483545068731108]
mean silhouette: 0.5466489287206502
sd silhouette: 0.0014811286182189163

[subject 0] overall=0.5456866208876072
clusters: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

[subject 1] overall=0.5459056584012325
clusters: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

[subject 2] overall=0.5483545068731108
clusters: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
